## Team Members

| # | Name | GitHub |
|---|------|--------|
| 1 | Rohima Ahmed Ibrahim | [rohimaahmed101-sys](https://github.com/rohimaahmed101-sys) |
| 2 | Shaimaa Kotit | [shaimaakotit-29](https://github.com/shaimaakotit-29) |
| 3 | Doha Mahmoud | [dohaalaraby-lang](https://github.com/dohaalaraby-lang) |
| 4 | Radwa Sabry | [RadwaSabry1](https://github.com/RadwaSabry1)|

**Dataset:** [Skin Cancer MNIST: HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000)

# AIE231 Course Project: Training-time feasibility check (D6)

## Why this notebook exists
The specification lists this as a dataset criterion (Section 1): *"One training run finishes in under 30 minutes on a free Google Colab GPU."*
This notebook **estimates** how long one training run takes for each candidate model, before real training starts in Phase 4. If a model is too slow, we change the setup now (image size, batch size) instead of finding out in Week 12.

## What it measures
| Model | What it is | Why it is timed |
|---|---|---|
| A | Small custom CNN from scratch | Model A of the image track |
| B-ResNet50 | ResNet-50, ImageNet weights, frozen base + new head | Model B option 1 |
| C-ResNet50 | Same, last block (`conv5`) unfrozen, small learning rate | Model C option 1 (heaviest) |
| B-VGG16 | VGG16, ImageNet weights, frozen base + new head | Model B option 2 |
| C-VGG16 | Same, last block (`block5`) unfrozen | Model C option 2 |

The spec allows **only** VGG16 or ResNet-50 as pretrained models; timing both lets the team choose with evidence.

## Important: this is an estimate, not a result
- These are **placeholder architectures** used only to measure speed. The real models are designed in Phase 4 (Model Architect).
- The images are a **random 70 % / 15 % sample for timing only**. This is **not** the project split. The real split (by `lesion_id`, stratified) is built in Phase 2 in `src/data.py`.
- No accuracy or F1 from this notebook is reported anywhere. Only the time per epoch is used.
- The official D6 evidence is the `train_time_sec` of real runs in `experiments/log.csv` (Phase 4).

## How to run
1. **Runtime → Change runtime type → T4 GPU → Save.** (Without a GPU the numbers are meaningless; the GPU check cell will stop the notebook.)
2. The `KAGGLE_API_TOKEN` secret from the Phase 1 notebook is reused.
3. **Runtime → Run all.** Takes roughly 15–25 minutes.
4. Send a screenshot of the **final table** and of the **package versions** cell, and download `timing_results.json`.

## Step 1: Kaggle credentials
Same as the Phase 1 notebook: the token is read from **Colab Secrets** (`KAGGLE_API_TOKEN`) and never written in the notebook.

In [1]:
# Step 1: load the Kaggle token from Colab Secrets
import os
from pathlib import Path
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print("Kaggle token loaded from Colab Secrets.")

Kaggle token loaded from Colab Secrets.


## Step 2: Download the dataset
Uses the Colab cache if the dataset was already downloaded in this session.

In [2]:
# Step 2: download HAM10000
!pip install -q --upgrade kagglehub
import kagglehub

DATA_DIR = Path(kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000"))
print("Dataset folder:", DATA_DIR)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Dataset folder: /kaggle/input/skin-cancer-mnist-ham10000


## Step 3: GPU check and package versions
1. Confirms TensorFlow sees a GPU and prints which one Colab assigned (T4 is the usual free GPU). If no GPU is found, the notebook stops here.
2. Prints the exact package versions in `requirements.txt` format. **These lines go into the repository's `requirements.txt` (checklist item R4).**

In [3]:
# Step 3: GPU check + package versions
import subprocess, sys
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
GPU_NAME = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print("GPU:", GPU_NAME)
print("TensorFlow:", tf.__version__)
print()

from importlib.metadata import version
PACKAGES = ["tensorflow", "numpy", "pandas", "scikit-learn", "matplotlib", "pillow", "pyyaml", "kagglehub"]
print("# requirements.txt  (Python " + sys.version.split()[0] + ", Google Colab)")
for pkg in PACKAGES:
    print(f"{pkg}=={version(pkg)}")

GPU: Tesla T4
TensorFlow: 2.21.0

# requirements.txt  (Python 3.13.16, Google Colab)
tensorflow==2.21.0
numpy==2.4.6
pandas==2.2.3
scikit-learn==1.6.1
matplotlib==3.10.0
pillow==11.3.0
pyyaml==6.0.3
kagglehub==1.0.3


## Step 4: Settings and image list
- Image size 224 × 224, batch size 32 (the planned reference values; tuning in Phase 4 may change them).
- Each image is taken **once** from the `HAM10000_images_part_*` folders (the Kaggle copy stores every image twice).
- A random 70 % (train) and 15 % (validation) sample, seed 42, only for timing.

In [4]:
# Step 4: settings + timing sample
import gc, json, time
import numpy as np
import pandas as pd

SEED = 42
IMG_SIZE = 224
BATCH = 32
EPOCHS_TIMED = 2           # epoch 1 = warm-up (reads images, fills cache, builds graph); epoch 2 = steady speed
LIMIT_SEC = 30 * 60        # the spec's 30-minute limit
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]   # same order as configs/base.yaml

tf.keras.utils.set_random_seed(SEED)

meta = pd.read_csv(next(p for p in DATA_DIR.rglob("*.csv") if p.name.lower() == "ham10000_metadata.csv"))
paths = {}
for p in sorted(DATA_DIR.glob("HAM10000_images_part_*/*.jpg")):
    paths.setdefault(p.stem, str(p))
meta = meta[meta["image_id"].isin(paths)].copy()
meta["path"] = meta["image_id"].map(paths)
meta["label"] = meta["dx"].map({c: i for i, c in enumerate(CLASSES)})

shuffled = meta.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_train = int(0.70 * len(shuffled))
n_val = int(0.15 * len(shuffled))
time_train = shuffled.iloc[:n_train]
time_val = shuffled.iloc[n_train:n_train + n_val]
print(f"Images found: {len(meta)}  |  timing train sample: {len(time_train)}  |  timing val sample: {len(time_val)}")
print("Reminder: this sample is for timing only, not the project split.")

Images found: 10015  |  timing train sample: 7010  |  timing val sample: 1502
Reminder: this sample is for timing only, not the project split.


## Step 5: Input pipeline (`tf.data`)
Reads each JPEG, resizes to 224 × 224, and **caches the resized images in RAM as uint8** (about 1 GB for the train sample). From epoch 2 onward no image is read from disk again, which is how the real training will run.
The model-specific pixel scaling is applied after the cache, so the same cached images serve all five models.

In [5]:
# Step 5: tf.data pipeline
AUTOTUNE = tf.data.AUTOTUNE

def load_resized(path, label):
    # Read a JPEG, resize to IMG_SIZE x IMG_SIZE, keep as uint8 to save RAM in the cache
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8), tf.one_hot(label, len(CLASSES))

def base_dataset(df):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values))
    return ds.map(load_resized, num_parallel_calls=AUTOTUNE).cache()

train_base = base_dataset(time_train)
val_base = base_dataset(time_val)

def batched(ds, preprocess, shuffle):
    # Apply a model's pixel scaling after the cache, then batch
    if shuffle:
        ds = ds.shuffle(2048, seed=SEED)
    ds = ds.map(lambda x, y: (preprocess(tf.cast(x, tf.float32)), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

print("Pipeline ready.")

Pipeline ready.


## Step 6: The five timing models
- Every model has the same head: global average pooling → Dropout(0.3) → Dense(7, softmax).
- Two light augmentations (random flip, random rotation) are inside each model, as in real training; Keras applies them only during training.
- **C models:** only the last block is unfrozen; BatchNormalization layers stay frozen (the usual practice when fine-tuning), learning rate 1e-5.
- These architectures are **placeholders for timing**, not the Phase 4 designs.

In [6]:
# Step 6: model builders
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.applications import resnet50, vgg16

def augment():
    # Fresh augmentation block per model (active only during training)
    return keras.Sequential([layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.1)], name="augment")

def head(x):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    return layers.Dense(len(CLASSES), activation="softmax")(x)

def custom_cnn():
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augment()(inputs)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    return keras.Model(inputs, head(x), name="A_custom_cnn")

def pretrained(kind, finetune):
    Base = ResNet50 if kind == "resnet50" else VGG16
    last_block = "conv5_" if kind == "resnet50" else "block5_"
    base = Base(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = finetune
    if finetune:
        for layer in base.layers:
            layer.trainable = layer.name.startswith(last_block) and not isinstance(layer, layers.BatchNormalization)
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augment()(inputs)
    x = base(x, training=False)    # keeps BatchNorm in inference mode
    name = ("C_" if finetune else "B_") + kind + ("_finetune" if finetune else "_frozen")
    return keras.Model(inputs, head(x), name=name)

MODELS = [
    # (label, builder, pixel preprocessing, learning rate)
    ("A  Custom CNN",          lambda: custom_cnn(),                   lambda x: x / 255.0,         1e-3),
    ("B  ResNet-50 frozen",    lambda: pretrained("resnet50", False),  resnet50.preprocess_input,   1e-3),
    ("C  ResNet-50 fine-tune", lambda: pretrained("resnet50", True),   resnet50.preprocess_input,   1e-5),
    ("B  VGG16 frozen",        lambda: pretrained("vgg16", False),     vgg16.preprocess_input,      1e-3),
    ("C  VGG16 fine-tune",     lambda: pretrained("vgg16", True),      vgg16.preprocess_input,      1e-5),
]
print("Models defined:", [m[0] for m in MODELS])

Models defined: ['A  Custom CNN', 'B  ResNet-50 frozen', 'C  ResNet-50 fine-tune', 'B  VGG16 frozen', 'C  VGG16 fine-tune']


## Step 7: Time 2 epochs per model
For each model: build it, train for 2 epochs on the timing sample (each epoch includes the validation pass, as in real training), and record the wall-clock time of each epoch.
- **Epoch 1** includes one-time costs (reading every image from disk, filling the cache, compiling the graph).
- **Epoch 2** is the steady speed that every later epoch will have.

In [7]:
# Step 7: timing loop
class EpochTimer(keras.callbacks.Callback):
    # Records wall-clock seconds per epoch (training + validation)
    def on_train_begin(self, logs=None):
        self.times = []
    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._t)

results = []
for label, build, prep, lr in MODELS:
    keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    model = build()
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="categorical_crossentropy")
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    timer = EpochTimer()
    print(f"\n=== {label}  (trainable parameters: {trainable:,}) ===")
    model.fit(batched(train_base, prep, True), validation_data=batched(val_base, prep, False),
              epochs=EPOCHS_TIMED, callbacks=[timer], verbose=2)
    e1, e2 = timer.times[0], timer.times[1]
    max_epochs = int((LIMIT_SEC - e1) // e2) + 1 if e1 < LIMIT_SEC else 0
    results.append({"model": label.split("  ")[1], "id": label.split("  ")[0], "trainable_params": trainable,
                    "epoch1_sec": round(e1, 1), "epoch2_sec": round(e2, 1),
                    "est_30_epochs_min": round((e1 + 29 * e2) / 60, 1),
                    "max_epochs_in_30_min": max_epochs})
    del model
print("\nTiming finished.")


=== A  Custom CNN  (trainable parameters: 94,151) ===
Epoch 1/2
220/220 - 81s - 368ms/step - loss: 1.1059 - val_loss: 1.0320
Epoch 2/2
220/220 - 10s - 47ms/step - loss: 1.0417 - val_loss: 1.0089
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== B  ResNet-50 frozen  (trainable parameters: 14,343) ===
Epoch 1/2
220/220 - 46s - 208ms/step - loss: 1.0212 - val_loss: 0.7479
Epoch 2/2
220/220 - 33s - 152ms/step - loss: 0.7658 - val_loss: 0.6474

=== C  ResNet-50 fine-tune  (trainable parameters: 14,967,815) ===
Epoch 1/2
220/220 - 54s - 246ms/step - loss: 1.0237 - val_loss: 0.7457
Epoch 2/2
220/220 - 42s - 192ms/step - loss: 0.7254 - val_loss: 0.6286
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== B  VGG16 frozen  (trainable parameters: 3,591) ===
Epoch 1/2
220/220 - 49s - 221ms/step - loss: 1.6934 - val_loss: 1.0005
Epoch 2/2
220/220 - 39s - 175ms/step - loss: 1.0692 - val_loss: 0.8347

=== C  VGG16 fine-tune  (trainable parameters: 7,083,015) ===
Epoch 1/2
220/220 - 47s - 21

## Step 8: Results and verdict
- **est. 30 epochs (min)** = epoch 1 + 29 × epoch 2. 30 epochs is a generous upper cap: with EarlyStopping on validation loss, real runs usually stop earlier.
- **max epochs in 30 min** = how many epochs fit inside the spec's limit.
- If Model C is trained in two stages (first the frozen head as in B, then fine-tuning), one "run" is B + C; the last lines show that combined estimate for 15 + 15 epochs.

In [8]:
# Step 8: results table + JSON
table = pd.DataFrame(results)[["id", "model", "trainable_params", "epoch1_sec", "epoch2_sec",
                               "est_30_epochs_min", "max_epochs_in_30_min"]]
table["under_30_min_at_30_epochs"] = np.where(table["est_30_epochs_min"] < 30, "yes", "NO")
print("GPU:", GPU_NAME, "| image size:", IMG_SIZE, "| batch:", BATCH, "| train sample:", len(time_train))
print(table.to_string(index=False))

def two_stage(b_id, c_id):
    b = next(r for r in results if r["model"] == b_id)
    c = next(r for r in results if r["model"] == c_id)
    return round((b["epoch1_sec"] + 14 * b["epoch2_sec"] + c["epoch1_sec"] + 14 * c["epoch2_sec"]) / 60, 1)

print()
print("Two-stage estimate (15 epochs frozen + 15 epochs fine-tune):")
print("  ResNet-50:", two_stage("ResNet-50 frozen", "ResNet-50 fine-tune"), "min")
print("  VGG16:    ", two_stage("VGG16 frozen", "VGG16 fine-tune"), "min")

out = {"gpu": GPU_NAME, "tensorflow": tf.__version__, "image_size": IMG_SIZE, "batch": BATCH,
       "train_sample": len(time_train), "val_sample": len(time_val), "results": results,
       "note": "Estimates from 2 timed epochs on a random timing sample; not the project split; no metrics reported."}
Path("timing_results.json").write_text(json.dumps(out, indent=2))
print("\nSaved:", Path("timing_results.json").resolve())

GPU: Tesla T4 | image size: 224 | batch: 32 | train sample: 7010
id               model  trainable_params  epoch1_sec  epoch2_sec  est_30_epochs_min  max_epochs_in_30_min under_30_min_at_30_epochs
 A          Custom CNN             94151        80.9        10.4                6.4                   167                       yes
 B    ResNet-50 frozen             14343        45.8        33.4               16.9                    53                       yes
 C ResNet-50 fine-tune          14967815        54.2        42.2               21.3                    42                       yes
 B        VGG16 frozen              3591        48.6        38.6               19.5                    46                       yes
 C     VGG16 fine-tune           7083015        47.2        39.6               19.9                    45                       yes

Two-stage estimate (15 epochs frozen + 15 epochs fine-tune):
  ResNet-50: 19.3 min
  VGG16:     19.8 min

Saved: /content/timing_results.json


## Next steps
1. Send a screenshot of the Step 3 versions and the Step 8 table, and download `timing_results.json`.
2. The versions go into `requirements.txt` (R4).
3. If every model fits under 30 minutes, D6 is supported now and confirmed by real runs in Phase 4. If not, we reduce image size or change batch size before Phase 4.
4. Commit this notebook to `notebooks/` with its outputs.